# LeetCode #41: First Missing Positive

https://leetcode.com/problems/first-missing-positive/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \log n)$ | $O(1)$ |
| **Hash Set** | $O(n)$ | $O(n)$ |
| **Optimal: In-place Index Negation ★** | $O(n)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Sort the array, then scan for the first gap in the positive integers starting from 1. Correct, but sorting costs $O(n \log n)$ and modifies the input.

### Hash Set
Insert every number into a hash set, then iterate $1, 2, 3, \ldots$ until a miss. $O(n)$ time but requires $O(n)$ extra space.

### Optimal: In-place Index Negation ★
Treat the array itself as a hash table: indices $0 \ldots n-1$ represent candidates $1 \ldots n$. In three linear passes — neutralize out-of-range values, negate `nums[|val|-1]` for each in-range value, then scan for the first positive slot — every presence is recorded without extra space.

**Why this is better than Hash Set:** Same $O(n)$ time, but $O(1)$ space by repurposing sign bits of the input array.

**Constraints:**
* $1 \leq n \leq 10^5$
* $-2^{31} \leq \text{nums}[i] \leq 2^{31} - 1$


## Solutions

### C#

In [ ]:
public class Solution {
    public int FirstMissingPositive(int[] nums) {
        int n = nums.Length;
        // Phase 1: neutralise values outside [1, n] so only valid candidates remain
        for (int i = 0; i < n; i++)
            if (nums[i] <= 0 || nums[i] > n)
                nums[i] = n + 1;

        // Phase 2: negate nums[val-1] to stamp "val is present" into the array
        for (int i = 0; i < n; i++) {
            int val = Math.Abs(nums[i]);
            if (val <= n)
                nums[val - 1] = -Math.Abs(nums[val - 1]);
        }

        // Phase 3: first positive slot means index+1 was never stamped
        for (int i = 0; i < n; i++)
            if (nums[i] > 0)
                return i + 1;

        return n + 1;
    }
}

### Python

In [ ]:
class Solution:
    def firstMissingPositive(self, nums: list[int]) -> int:
        n = len(nums)
        # Replace out-of-range values so only [1, n] candidates remain
        for i in range(n):
            if nums[i] <= 0 or nums[i] > n:
                nums[i] = n + 1

        # Negate index (val-1) to signal that val is present in the array
        for i in range(n):
            val = abs(nums[i])
            if val <= n:
                nums[val - 1] = -abs(nums[val - 1])

        # First non-negative slot tells us which positive integer is missing
        for i in range(n):
            if nums[i] > 0:
                return i + 1

        return n + 1

### Go

In [ ]:
func firstMissingPositive(nums []int) int {
    n := len(nums)
    // Neutralise values outside [1, n] — they can't be the answer
    for i := range nums {
        if nums[i] <= 0 || nums[i] > n {
            nums[i] = n + 1
        }
    }

    // Negate position (val-1) to record that val exists in the array
    for i := range nums {
        val := nums[i]
        if val < 0 {
            val = -val
        }
        if val <= n && nums[val-1] > 0 {
            nums[val-1] = -nums[val-1]
        }
    }

    // Scan for the first slot still positive; that index+1 was never seen
    for i := range nums {
        if nums[i] > 0 {
            return i + 1
        }
    }
    return n + 1
}

### Rust

In [ ]:
impl Solution {
    pub fn first_missing_positive(nums: Vec<i32>) -> i32 {
        let mut nums = nums;
        let n = nums.len() as i32;

        // Replace values outside [1, n] with a harmless sentinel
        for i in 0..nums.len() {
            if nums[i] <= 0 || nums[i] > n {
                nums[i] = n + 1;
            }
        }

        // Negate index (val-1) to stamp val's presence into the array itself
        for i in 0..nums.len() {
            let val = nums[i].abs();
            if val <= n {
                let idx = (val - 1) as usize;
                nums[idx] = -nums[idx].abs();
            }
        }

        // First positive entry means its index+1 was never seen
        for i in 0..nums.len() {
            if nums[i] > 0 {
                return i as i32 + 1;
            }
        }
        n + 1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums = [1, 2, 0]`
After neutralising: `[1, 2, 4]`. Negation marks indices 0 and 1. Index 2 stays positive → return **3**.

### 2. Slightly Complex
**Input:** `nums = [3, 4, -1, 1]`
After neutralising: `[3, 4, 5, 1]`. Negation marks indices 0 and 2. First positive is index 1 → return **2**.

### 3. Edge Case: Time Factor
**Input:** `nums = [2, 3, 4, ..., n, 1]` (all $n$ values in $[1, n]$ perfectly present)
Every index gets negated in phase 2; phase 3 scans all $n$ slots before reaching index $n$ (which doesn't exist), so we return $n + 1$. This is the maximum-work case for all three passes, confirming $O(n)$.

### 4. Edge Case: Space Factor
**Input:** `nums = [2]` ($n = 1$)
After neutralising: `[2]` → 2 > 1, so it stays as $n+1 = 2$ sentinel. No negation fires. Index 0 is positive → return **1**. The $O(1)$ space bound holds even at minimal array size.

### 5. Almost-Impossible but Plausible
**Input:** `nums = [1, 1]` (duplicate 1s)
Both elements try to negate index 0; both find it already negative after the first, so $\lvert\text{nums}[0]\rvert = 1$ each time. Index 1 stays positive → return **2**. Deduplication is implicit — no extra data structure needed.
